# Part 2: How Real VLAs Represent Actions

## Notebook 6 — π₀ (pi0): Flow Matching + Action Expert

pi0 (Physical Intelligence, 2024) is a VLA that augments a pre-trained PaliGemma VLM with a dedicated **action expert**. Actions are generated via **flow matching** — a continuous ODE-based approach related to diffusion.

We load pi0 from leRobot and inspect its action expert and flow matching pipeline.


### 1. Load pi0 configuration

pi0 has an explicit `action_expert_variant` — a separate Gemma model dedicated solely to generating actions.


In [ ]:
from lerobot.policies.pi0.configuration_pi0 import PI0Config

cfg = PI0Config()
print(f"Policy type: pi0 (Flow Matching VLA)")
print(f"Action expert:       {cfg.action_expert_variant}")  # gemma_300m
print(f"Chunk size:          {cfg.chunk_size}")  # 50
print(f"Action steps:        {cfg.n_action_steps}")  # 50
print(f"Max action dim:      {cfg.max_action_dim}")  # 32 (padded)
print(f"Max state dim:       {cfg.max_state_dim}")  # 32
print(f"Train expert only:   {cfg.train_expert_only}")  # False
print(f"Tokenizer max len:   {cfg.tokenizer_max_length}")  # 48


### 2. Architecture: VLM + Action Expert

pi0 uses a **mixture of experts** architecture:
- PaliGemma VLM (SigLIP + Gemma 2B) — handles vision + language
- Action expert (Gemma 300M) — handles action generation

The action expert is a separate transformer with its own weights. It receives the VLM's processed observation embeddings and generates continuous action vectors.


In [ ]:
# pi0 Architecture (conceptual)
print("pi0 Architecture:")
print("┌─────────────────────────────────────────────┐")
print("│  PaliGemma VLM (SigLIP + Gemma 2B)          │")
print("│  ┌─────────┐  ┌──────────────────────┐      │")
print("│  │ SigLIP   │  │ Gemma 2B Backbone    │      │")
print("│  │ (Vision) │  │ (Language + Fusion)  │      │")
print("│  └────┬─────┘  └──────────┬───────────┘      │")
print("│       │                   │                  │")
print("│       └───────┬───────────┘                  │")
print("│               ▼                              │")
print("│  ┌──────────────────────────────────────┐    │")
print("│  │  Action Expert (Gemma 300M)           │    │")
print("│  │  • action_in_proj  (action_dim→width) │    │")
print("│  │  • action_out_proj (width→action_dim) │    │")
print("│  │  • Flow matching timestep MLP         │    │")
print("│  │  → Continuous action chunk (50 × 7)   │    │")
print("│  └──────────────────────────────────────┘    │")
print("└─────────────────────────────────────────────┘")


### 3. Flow Matching: how actions are generated

Flow matching learns a continuous transformation (flow) from a simple distribution (e.g., Gaussian) to the action distribution.

During training: given a real action a₁, sample noise a₀ ~ N(0,I), interpolate a_t = (1-t)·a₀ + t·a₁, train model to predict velocity da/dt.

During inference: sample a₀ ~ N(0,I), integrate the learned velocity field to get a₁ (the action chunk).


In [ ]:
# Flow Matching vs Diffusion
print("Flow Matching (pi0):")
print("  - Learns a vector field v(t, x) that maps noise → data")
print("  - Training: predict velocity da/dt at interpolated points")
print("  - Inference: ODE integration (Euler/RK4), typically 10 steps")
print("  - Deterministic or stochastic depending on noise schedule")

print("Diffusion (Diffusion Policy):")
print("  - Learns to predict noise ε added at timestep t")
print("  - Training: predict noise from noisy action")
print("  - Inference: iterative denoising, typically 50-100 steps")
print("  - Stochastic by design")

print("Key difference: Flow matching typically needs fewer inference steps.")


### 4. The action expert — what it actually does

The action expert is a Gemma transformer specialized for action generation. Key layers from the leRobot source:


In [ ]:
# From modeling_pi0.py:
# self.action_in_proj = nn.Linear(max_action_dim, action_expert.width)
# self.action_out_proj = nn.Linear(action_expert.width, max_action_dim)
# self.state_proj = nn.Linear(max_state_dim, action_expert.width)
# self.action_time_mlp_in = nn.Linear(2 * width, width)
# self.action_time_mlp_out = nn.Linear(width, width)

# The action expert takes:
#   - The noisy action chunk (flow matching timestep)
#   - The VLM's fused observation embeddings
#   - The robot's proprioceptive state
# And outputs continuous action predictions

print("Action Expert Data Flow:")
print("  state → state_proj → [B, width]")
print("  action → action_in_proj → [B, chunk, width]")
print("  time → action_time_mlp → [B, chunk, width]")
print("  All combined → Gemma expert → action_out_proj → [B, chunk, action_dim]")


### 5. train_expert_only: fine-tuning strategy

pi0 supports freezing the VLM backbone and training only the action expert. This preserves internet-scale visual/language knowledge while adapting actions.


In [ ]:
# Key config flag
print(f"train_expert_only: {cfg.train_expert_only}")

print("When train_expert_only=True:")
print("  ✓ Action expert weights are updated")
print("  ✗ VLM backbone (SigLIP + Gemma 2B) is frozen")
print("  → Preserves general knowledge, only adapts actions")
print("  → Similar philosophy to LoRA but structural rather than low-rank")


### In Short

pi0 keeps actions **continuous** and uses an **explicit action expert** — a separate transformer specialized for generating smooth action trajectories via flow matching. The action expert is a dedicated module built for control, not a repurposed language head.
